# Foundations · FastAPI

Your service is an **HTTP API**: programs send it a request and get a response. **FastAPI** is a Python framework for writing one. You declare routes as functions and the shapes of requests and responses as Pydantic models; FastAPI does the parsing, validation, error messages and documentation.

An HTTP exchange has five parts:

| Part | Example |
|---|---|
| Method | `POST` (send data), `GET` (read) |
| Path | `/v1/classify` |
| Headers | `Content-Type: application/json`, `X-Request-ID: 42` |
| Body | `{"id": "t1", "text": "My brake is broken"}` |
| Status code | `200` OK, `401` not authenticated, `413` too large, `422` invalid input, `429` slow down, `500` server bug, `503` not ready |

The status code is part of your contract. The harness checks it as carefully as the body.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## 1 · A complete API in ten lines

`TestClient` calls the app directly, in this process: no server, no port, no network. It is how our tests and the harness's `--in-process` mode work.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

demo = FastAPI()


class FareRequest(BaseModel):
    minutes: int = Field(ge=0, le=1440, description="Ride length in minutes")
    ebike: bool = False


class FareResponse(BaseModel):
    total: float


@demo.post("/fare")
def fare(body: FareRequest) -> FareResponse:
    rate = 0.40 if body.ebike else 0.20
    return FareResponse(total=round(1.25 + rate * body.minutes, 2))


client = TestClient(demo)
response = client.post("/fare", json={"minutes": 30, "ebike": True})
print(response.status_code, response.json())

## 2 · Validation is free

Send bad input. You did not write a single `if`: the Pydantic model is the validation, and FastAPI answers `422` with a precise message.

In [ ]:
for bad in [{"minutes": -5}, {"minutes": "half an hour"}, {}]:
    r = client.post("/fare", json=bad)
    print(r.status_code, r.json()["detail"][0]["msg"])

## 3 · `def` or `async def`

- `def` routes run in a thread pool. Fine for quick work.
- `async def` routes run on the event loop. While one request **waits** (for a model, a database, another API), the loop serves others.

Our routes are `async def` because nearly all their time is spent waiting for a model. The one rule: inside `async def`, never call something slow that is not awaited, or every request waits behind it.

## 4 · The course service, in-process

The same technique works on the real service. Offline mode replaces the models with deterministic stand-ins, so this cell needs no Azure.

In [ ]:
from app.config import LLMMode, Settings
from app.main import create_app

# service_api_key=None: if you set the Week 10 key in your shell, it does not apply here.
offline = Settings(llm_mode=LLMMode.OFFLINE, student_id="demo", service_api_key=None, _env_file=None)
with TestClient(create_app(offline)) as service:
    print(service.get("/v1/info").json())
    print(service.post("/v1/classify", json={"id": "t1", "text": "I was charged twice"}).json())
    routes = service.get("/openapi.json").json()["paths"]
    print("Routes:", ", ".join(routes))

## 5 · A real server

To run the service as a server, open a terminal in `fy26/` and run:

```bash
uv run uvicorn app.main:app --reload
```

Then open http://localhost:8000/docs: an interactive page generated from your models, where you can call every route. `--reload` restarts the server when you save a file. Stop it with Ctrl+C.

## 6 · Exercise

Add a `GET /stations/{station_id}` route to `demo` that returns `app.domain.get_station(station_id)`, and answers **404** when the station does not exist (the function returns a dict with an `error` key). Hint: `from fastapi import HTTPException` and `raise HTTPException(status_code=404, detail=...)`. Then run the check cell.

In [ ]:
# Your route here.

In [ ]:
check = TestClient(demo)
found = check.get("/stations/LS-05")
assert found.status_code == 200, f"GET /stations/LS-05 returned {found.status_code}. Write the route in the cell above."
assert found.json()["name"] == "Central Station"
assert check.get("/stations/LS-99").status_code == 404, "An unknown station must return 404."
print("Exercise passed.")

## What to remember

- A route is a function; its parameters and return type are the contract.
- Pydantic models validate input for you and document the API at `/docs`.
- Status codes are part of the contract: 4xx means the caller's fault, 5xx means yours.
- `TestClient` tests the whole app without a server.

**Learn more:** [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/) · [FastAPI release notes](https://fastapi.tiangolo.com/release-notes/)